<a href="https://colab.research.google.com/github/Maanvijangir/Stock_market_forecasting/blob/main/stock_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install vaderSentiment

In [ ]:
import pandas as pd

NEWS API

In [ ]:
!pip install newsapi-python

In [ ]:
from newsapi import NewsApiClient
API_KEY = "626d8eae3efe49a18e2028f9c1557a41"
newsapi = NewsApiClient(api_key=API_KEY)

In [ ]:
articles = newsapi.get_everything(
    q="TCS",
    language="en",
    sort_by="publishedAt",
    page_size=10
)
print(len(articles["articles"]))

10


In [ ]:
print(articles["articles"][0]["title"])

Marathons generate $6 billion for cities, TCS leads the charge


Headlines

In [ ]:
headlines = []
for article in articles["articles"]:
  headlines.append(article["title"])
headlines[:5]

['Marathons generate $6 billion for cities, TCS leads the charge',
 '2026 TCS New York City Marathon kicks off with NYRR Opening Ceremony',
 'NYRR gift 400 pairs of shoes to NYC students ahead of TCS NYC Marathon',
 'Comedian Tracy Morgan is grand marshal of 2026 NYC Marathon',
 "The NYC Marathon Inspired Me to Escape Afghanistan. Now I'm Running the Race"]

In [ ]:
stocks = {
    "TCS.NS": "TCS",
    "INFY.NS": "Infosys",
    "RELIANCE.NS": "Reliance Industries",
    "HDFCBANK.NS": "HDFC Bank",
    "JIOFIN.NS": "Jio Financial Services"
}

In [ ]:
def get_stock_news(company):
  articles = newsapi.get_everything(
      q=company,
      language="en",
      sort_by = "publishedAt",
      page_size=20
      )
  news_list = []
  for article in articles["articles"]:
    news_list.append({
        "Headline": article["title"],
        "Source": article["source"]["name"]
    })
  return pd.DataFrame(news_list)

In [ ]:
df_tcs = get_stock_news("TCS")
df_tcs.head()

,Headline,Source
0,"Marathons generate $6 billion for cities, TCS ...",Nextbigwhat.com
1,2026 TCS New York City Marathon kicks off with...,WABC-TV
2,NYRR gift 400 pairs of shoes to NYC students a...,WABC-TV
3,Comedian Tracy Morgan is grand marshal of 2026...,New York Daily News
4,The NYC Marathon Inspired Me to Escape Afghani...,Teen Vogue


VADER Sentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

In [ ]:
headline = "TCS reports strong growth in quaterly earnings"
score = analyzer.polarity_scores(headline)
print(score)

{'neg': 0.0, 'neu': 0.459, 'pos': 0.541, 'compound': 0.7096}


In [ ]:
def get_sentiment(text):
  score = analyzer.polarity_scores(text)
  compound = score["compound"]
  if compound >= 0.05:
    return "Positive"
  elif compound <= -0.05:
    return "Negative"
  else:
    return "Neutral"

In [ ]:
df_news = get_stock_news("stocks")
df_news.head()

,Headline,Source
0,"Currency movements, taxation: Risks to know be...",Business Standard
1,What W&T Offshore (WTI) Said on Its Q3 Earning...,Biztoc.com
2,What SPX Technologies (SPXC) Said on Its Q3 Ea...,Biztoc.com
3,A look back at consumer discretionary - leisur...,Biztoc.com
4,INDIA bloc leaders to travel in open buses ahe...,The Times of India


In [ ]:
df_news["Sentiment"] = df_news["Headline"].apply(get_sentiment)
df_news.head()

,Headline,Source,Sentiment
0,"Currency movements, taxation: Risks to know be...",Business Standard,Negative
1,What W&T Offshore (WTI) Said on Its Q3 Earning...,Biztoc.com,Neutral
2,What SPX Technologies (SPXC) Said on Its Q3 Ea...,Biztoc.com,Neutral
3,A look back at consumer discretionary - leisur...,Biztoc.com,Neutral
4,INDIA bloc leaders to travel in open buses ahe...,The Times of India,Neutral


In [ ]:
df_tcs["Sentiment"] = df_tcs["Headline"].apply(get_sentiment)
df_tcs.head()

,Headline,Source,Sentiment
0,"Marathons generate $6 billion for cities, TCS ...",Nextbigwhat.com,Neutral
1,2026 TCS New York City Marathon kicks off with...,WABC-TV,Neutral
2,NYRR gift 400 pairs of shoes to NYC students a...,WABC-TV,Positive
3,Comedian Tracy Morgan is grand marshal of 2026...,New York Daily News,Positive
4,The NYC Marathon Inspired Me to Escape Afghani...,Teen Vogue,Positive


Sentiment Summary

In [ ]:
def sentiment_summary(df):
  summary = df["Sentiment"].value_counts()
  positive = summary.get("Positive",0)
  neutral = summary.get("Neutral",0)
  negative = summary.get("Negative",0)

  total = len(df)

  positive_pct = round((positive/total)*100,2)
  neutral_pct = round((neutral/total)*100,2)
  negative_pct = round((negative/total)*100,2)

  return{
      "Positive %": positive_pct,
      "Neutral %": neutral_pct,
      "Negative %": negative_pct
  }

In [ ]:
sentiment_summary(df_news)

{'Positive %': np.float64(16.67),
 'Neutral %': np.float64(72.22),
 'Negative %': np.float64(11.11)}

In [ ]:
results = []
for ticker, company in stocks.items():
  df_news = get_stock_news(company)
  df_news["Sentiment"] = df_news["Headline"].apply(get_sentiment)
  summary = sentiment_summary(df_news)

  results.append({
      "Ticker": ticker,
      "Company": company,
      "Positive %": summary["Positive %"],
      "Neutral %": summary["Neutral %"],
      "Negative %": summary["Negative %"]
  })

In [ ]:
sentiment_df = pd.DataFrame(results)
sentiment_df

,Ticker,Company,Positive %,Neutral %,Negative %
0,TCS.NS,TCS,50.0,35.0,15.0
1,INFY.NS,Infosys,65.0,20.0,15.0
2,RELIANCE.NS,Reliance Industries,70.0,15.0,15.0
3,HDFCBANK.NS,HDFC Bank,75.0,15.0,10.0
4,JIOFIN.NS,Jio Financial Services,70.0,25.0,5.0


In [ ]:
def signal(row):
  if row["Positive %"] > row["Negative %"]:
    return "Bullish"
  elif row["Negative %"] > row["Postive %"]:
    return "Bearish"
  else:
    return "Neutral"

In [ ]:
sentiment_df["Signal"] = (sentiment_df.apply(signal, axis=1))

In [ ]:
sentiment_df

,Ticker,Company,Positive %,Neutral %,Negative %,Signal
0,TCS.NS,TCS,50.0,35.0,15.0,Bullish
1,INFY.NS,Infosys,65.0,20.0,15.0,Bullish
2,RELIANCE.NS,Reliance Industries,70.0,15.0,15.0,Bullish
3,HDFCBANK.NS,HDFC Bank,75.0,15.0,10.0,Bullish
4,JIOFIN.NS,Jio Financial Services,70.0,25.0,5.0,Bullish


In [ ]:
sentiment_df.to_csv("sentiment_results.csv",index=False)